In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_replace, col, split, when, avg
from pyspark.sql.types import IntegerType, DoubleType

# Start Spark Session
spark = SparkSession.builder.appName("LoanDataCleaning").getOrCreate()

# Load CSV
df = spark.read.csv("loansData.csv", header=True, inferSchema=True)


df.show(10, truncate=True)




+----------------+--------------------------+------+-----------+------------------+--------------------+-----+--------------+--------------+----------+-----------------+------------------------+------------------------------+-----------------+
|Amount.Requested|Amount.Funded.By.Investors|  rate|Loan.Length|      Loan.Purpose|Debt.To.Income.Ratio|State|Home.Ownership|Monthly.Income|FICO.Range|Open.CREDIT.Lines|Revolving.CREDIT.Balance|Inquiries.in.the.Last.6.Months|Employment.Length|
+----------------+--------------------------+------+-----------+------------------+--------------------+-----+--------------+--------------+----------+-----------------+------------------------+------------------------------+-----------------+
|           81174|                     20000| 8.90%|  36 months|debt_consolidation|              14.90%|   SC|      MORTGAGE|       6541.67|   735-739|               14|                   14272|                             2|         < 1 year|
|           99592|      

In [ ]:
df_clean = df.withColumn("clean_value", regexp_replace(col("rate"), "%", ""))

df_clean.show(truncate=False)

+----------------+--------------------------+------+-----------+------------------+--------------------+-----+--------------+--------------+----------+-----------------+------------------------+------------------------------+-----------------+-----------+
|Amount.Requested|Amount.Funded.By.Investors|rate  |Loan.Length|Loan.Purpose      |Debt.To.Income.Ratio|State|Home.Ownership|Monthly.Income|FICO.Range|Open.CREDIT.Lines|Revolving.CREDIT.Balance|Inquiries.in.the.Last.6.Months|Employment.Length|clean_value|
+----------------+--------------------------+------+-----------+------------------+--------------------+-----+--------------+--------------+----------+-----------------+------------------------+------------------------------+-----------------+-----------+
|81174           |20000                     |8.90% |36 months  |debt_consolidation|14.90%              |SC   |MORTGAGE      |6541.67       |735-739   |14               |14272                   |2                             |< 1 yea

In [ ]:
df_clean = df.withColumn("rate", regexp_replace(col("rate"), "%", ""))

df_clean.show(truncate=False)

+----------------+--------------------------+-----+-----------+------------------+--------------------+-----+--------------+--------------+----------+-----------------+------------------------+------------------------------+-----------------+
|Amount.Requested|Amount.Funded.By.Investors|rate |Loan.Length|Loan.Purpose      |Debt.To.Income.Ratio|State|Home.Ownership|Monthly.Income|FICO.Range|Open.CREDIT.Lines|Revolving.CREDIT.Balance|Inquiries.in.the.Last.6.Months|Employment.Length|
+----------------+--------------------------+-----+-----------+------------------+--------------------+-----+--------------+--------------+----------+-----------------+------------------------+------------------------------+-----------------+
|81174           |20000                     |8.90 |36 months  |debt_consolidation|14.90%              |SC   |MORTGAGE      |6541.67       |735-739   |14               |14272                   |2                             |< 1 year         |
|99592           |19200     

In [ ]:
df_clean=df.withColumn("Loan.Length",split(col("`Loan.Length`")," ")[0])

df_clean.show(truncate=False)

+----------------+--------------------------+------+-----------+------------------+--------------------+-----+--------------+--------------+----------+-----------------+------------------------+------------------------------+-----------------+
|Amount.Requested|Amount.Funded.By.Investors|rate  |Loan.Length|Loan.Purpose      |Debt.To.Income.Ratio|State|Home.Ownership|Monthly.Income|FICO.Range|Open.CREDIT.Lines|Revolving.CREDIT.Balance|Inquiries.in.the.Last.6.Months|Employment.Length|
+----------------+--------------------------+------+-----------+------------------+--------------------+-----+--------------+--------------+----------+-----------------+------------------------+------------------------------+-----------------+
|81174           |20000                     |8.90% |36         |debt_consolidation|14.90%              |SC   |MORTGAGE      |6541.67       |735-739   |14               |14272                   |2                             |< 1 year         |
|99592           |19200 

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

df_clean=df.withColumn("Loan.Length",split(col("`Loan.Length`")," ")[0])\
           .withColumn("Loan.Length_Type",split(col("`Loan.Length`")," ")[1])
df_clean.show(truncate=False)

IndentationError: unexpected indent (1830750768.py, line 5)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

df_clean = df.withColumn(
    "Loan.Length_Value",
    F.split(col("`Loan.Length`"), " ").getItem(0)
).withColumn(
    "Loan.Length_Type",
    F.split(col("`Loan.Length`"), " ").getItem(1)
)

df_clean.show(truncate=False)

+----------------+--------------------------+------+-----------+------------------+--------------------+-----+--------------+--------------+----------+-----------------+------------------------+------------------------------+-----------------+-----------------+----------------+
|Amount.Requested|Amount.Funded.By.Investors|rate  |Loan.Length|Loan.Purpose      |Debt.To.Income.Ratio|State|Home.Ownership|Monthly.Income|FICO.Range|Open.CREDIT.Lines|Revolving.CREDIT.Balance|Inquiries.in.the.Last.6.Months|Employment.Length|Loan.Length_Value|Loan.Length_Type|
+----------------+--------------------------+------+-----------+------------------+--------------------+-----+--------------+--------------+----------+-----------------+------------------------+------------------------------+-----------------+-----------------+----------------+
|81174           |20000                     |8.90% |36 months  |debt_consolidation|14.90%              |SC   |MORTGAGE      |6541.67       |735-739   |14          

In [ ]:
#number of loans in each category
df.groupBy("`Loan.Purpose`").count().orderBy("count", ascending = False).show()

+------------------+-----+
|      Loan.Purpose|count|
+------------------+-----+
|debt_consolidation| 1307|
|       credit_card|  444|
|             other|  201|
|  home_improvement|  152|
|    major_purchase|  101|
|    small_business|   87|
|               car|   50|
|           wedding|   39|
|           medical|   30|
|            moving|   29|
|          vacation|   21|
|             house|   20|
|       educational|   15|
|  renewable_energy|    4|
+------------------+-----+



In [ ]:
df.groupBy("`State`").sum("`Amount.Funded.By.Investors`").show()

+-----+-------------------------------+
|State|sum(Amount.Funded.By.Investors)|
+-----+-------------------------------+
|   SC|                         291250|
|   AZ|                         566525|
|   LA|                         387625|
|   MN|                         526725|
|   NJ|                        1076475|
|   DC|                         171625|
|   OR|                         331050|
|   VA|                        1008525|
|   RI|                         172400|
|   KY|                         227850|
|   WY|                          29300|
|   NH|                         251650|
|   MI|                         550825|
|   NV|                         375900|
|   WI|                         271950|
|   CA|                        5285675|
|   CT|                         608475|
|   MT|                          85700|
|   NC|                         729300|
|   VT|                          95800|
+-----+-------------------------------+
only showing top 20 rows


In [ ]:
#loan amount is greater than 10000
df.filter(df["`Amount.Funded.By.Investors`"]>10000).count()

1229

In [ ]:
#income is greater than 6000
df.filter(df["`Monthly.Income`"]>6000).count()

821